# Phase 2 - Part B: Generative AI Integration

## 1. Generative AI Model Setup

We use **LLaMA 3.3 70B** (Meta's open-source LLM) accessed through the **Groq API**.

**Why Groq + LLaMA?**
- Free API access (no credit card required)
- Open-source model aligns with academic transparency
- Fast inference via Groq's specialized hardware
- Listed as an option in the project handbook


In [8]:
# Import required libraries
import os
from dotenv import load_dotenv
from groq import Groq

# Load API key from .env file
load_dotenv()
api_key = os.getenv("GROQ_API_KEY")

# Initialize the Groq client
client = Groq(api_key=api_key)

# Model configuration
MODEL_NAME = "llama-3.3-70b-versatile"

# Quick connection test
response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[{"role": "user", "content": "Reply with: connection OK"}],
    max_tokens=10
)

print("Model:", MODEL_NAME)
print("Response:", response.choices[0].message.content)

Model: llama-3.3-70b-versatile
Response: connection OK


## 2. Prompt Template Design

### Template 1: Simple Explanation

**Goal:** Explain the prediction to the patient in plain, friendly language.

**Target user:** Patients with no medical background.

**Design choice:** Short response (3-5 sentences), no medical jargon, reassuring tone.

In [15]:
# Template 1: Simple Explanation
# Goal: explain results in plain language for non-medical users

template_1_system = """You are a friendly health assistant. 
Explain medical results in simple, everyday language. 
Avoid technical terms. Keep responses short (3-5 sentences).
Be reassuring but honest."""

template_1_user = """A patient received their liver health screening result.

Prediction: {prediction}
Key values:
{features}

Explain this result in plain language, as if talking to a friend with no medical background.
Focus on the overall meaning, not the numbers."""

In [16]:
def simple_explanation(prediction, features):
    """Generate a simple explanation using Template 1."""
    
    user_message = template_1_user.format(
        prediction=prediction,
        features=features
    )
    
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": template_1_system},
            {"role": "user", "content": user_message}
        ],
        temperature=0.7,
        max_tokens=300
    )
    
    return response.choices[0].message.content

In [17]:
import pandas as pd

# Load raw dataset (easier for the AI to interpret)
data = pd.read_csv("Raw_Dataset/indian_liver_patient.csv")

# 3 diverse test cases
test_indices = [0, 315, 440] 

for i, idx in enumerate(test_indices, start=1):
    patient = data.iloc[idx]
    
    # Convert prediction: 1 = Liver Disease, 2 = No Liver Disease
    prediction = "Liver Disease Detected" if patient["Dataset"] == 1 else "No Liver Disease"
    
    # Format features (drop the label so the AI doesn't see the answer)
    features = "\n".join([
        f"- {col}: {val}"
        for col, val in patient.drop("Dataset").items()
    ])
    
    # Generate explanation
    print(f"Patient {i}: (row {idx}), Prediction: {prediction}")
    
    result = simple_explanation(prediction, features)
    print(result)

Patient 1: (row 0), Prediction: Liver Disease Detected
Hey, I know this might be a bit concerning, but let's break it down. The screening result shows that there's a possibility of liver disease. This doesn't necessarily mean it's severe, but it's something we should look into further. We'll likely need to do some more tests to understand what's going on and figure out the best next steps to take care of your liver health.
Patient 2: (row 315), Prediction: No Liver Disease
Your liver health screening result looks great. It says you don't have any liver disease, which is fantastic news. Your liver is working properly and everything seems to be in order. You can breathe a sigh of relief and just keep taking care of yourself as you normally do. Overall, your liver is healthy and that's something to be happy about.
Patient 3: (row 440), Prediction: Liver Disease Detected
Hey, I know getting test results can be nerve-wracking. Unfortunately, your liver screening showed that you might have s

### Template 2: Personalized Explanation

**Goal:** Provide a personalized explanation by linking the prediction to the patient’s specific values.

**Target user:** Patients who want a more relevant and tailored explanation of their results.

**Design choice:** Uses the patient’s actual data (e.g., age, lab values) to make the explanation more meaningful and engaging, while still keeping the language simple and easy to understand. Maintains a supportive tone without using complex medical terminology.

**Implementation note:** Due to the length of personalized responses, outputs for this template were saved directly to text files instead of being displayed in the notebook. This ensured that full responses were preserved without truncation and allowed for more accurate evaluation.

In [11]:
# Template 2: Personalized Explanation
# Goal: explain the result using the patient's specific values

template_2_system = """You are a personalized medical assistant.
Use the patient's specific data in your explanation.
Mention relevant patient values only if they help explain the prediction.
Explain what these values may indicate in simple language.
Keep explanations simple without unnecessary medical complexity.
Maintain a supportive and reassuring tone."""

template_2_user = """A patient received their liver health screening result.

Prediction: {prediction}
Key values:
{features}

Explain this result by directly referring to the patient’s specific values.
Highlight any notable indicators in a simple and personalized way."""

def personalized_explanation(prediction, features):
    """Generate a personalized explanation using Template 2."""
    
    user_message = template_2_user.format(
        prediction=prediction,
        features=features
    )
    
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": template_2_system},
            {"role": "user", "content": user_message}
        ],
        temperature=0.5,
        max_tokens=700
    )
    
    return response.choices[0].message.content

In [14]:
import os

output_dir = "Generative_AI/example_outputs"
os.makedirs(output_dir, exist_ok=True)

for i, idx in enumerate(test_indices, start=1):
    patient = data.iloc[idx]

    prediction = "Liver Disease Detected" if patient["Dataset"] == 1 else "No Liver Disease"

    features = "\n".join([
        f"- {col}: {val}"
        for col, val in patient.drop("Dataset").items()
    ])

    result = personalized_explanation(prediction, features)

    file_path = f"{output_dir}/template_2_patient_{i}.txt"

    with open(file_path, "w", encoding="utf-8") as f:
        f.write(f"Patient {i}: (row {idx}), Prediction: {prediction}\n\n")
        f.write(result)

    print(f"Saved: {file_path}")

Saved: Generative_AI/example_outputs/template_2_patient_1.txt
Saved: Generative_AI/example_outputs/template_2_patient_2.txt
Saved: Generative_AI/example_outputs/template_2_patient_3.txt


### Template 3: Detailed Reasoning

**Goal:** Provide a structured explanation of the prediction by breaking down key medical indicators and explaining their role in the result.

**Target user:** Users who want a deeper understanding of how specific lab values contribute to the prediction.

**Design choice:** Focuses on explaining important features in a structured way (what it is, what it indicates, and its relevance to liver health). This improves transparency and helps users understand the reasoning behind the prediction without overwhelming them with unnecessary details.

**Implementation note:** This template focuses on structured reasoning, so responses are more detailed than simple explanations but still limited to the most relevant features to maintain clarity.

In [18]:
# Template 3: Detailed Reasoning

template_3_system = """You are a medical assistant with clinical knowledge.
Provide structured and clear explanations.
Explain relevant medical features based on the patient data.

For important indicators, include:
1. What it is (simple meaning)
2. What it indicates
3. Its relevance to liver health

Keep explanations clear and easy to follow.
Avoid unnecessary details.
Focus only on features that significantly impact the prediction."""

template_3_user = """A patient received their liver health screening result.

Prediction: {prediction}
Key values:
{features}

Analyze the results and:

- Identify the most relevant features from the data
- Explain each important feature in simple terms
- Highlight abnormal or notable values
- Link these values to the prediction
- Provide a final summary explaining why the prediction was made
"""

def detailed_reasoning(prediction, features):
    user_message = template_3_user.format(
        prediction=prediction,
        features=features
    )

    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": template_3_system},
            {"role": "user", "content": user_message}
        ],
        temperature=0.4,
        max_tokens=600
    )

    return response.choices[0].message.content

In [19]:
import os

output_dir = "Generative_AI/example_outputs"
os.makedirs(output_dir, exist_ok=True)

for i, idx in enumerate(test_indices, start=1):
    patient = data.iloc[idx]

    prediction = "Liver Disease Detected" if patient["Dataset"] == 1 else "No Liver Disease"

    features = "\n".join([
        f"- {col}: {val}"
        for col, val in patient.drop("Dataset").items()
    ])

    result = detailed_reasoning(prediction, features)

    file_path = f"{output_dir}/template_3_patient_{i}.txt"

    with open(file_path, "w", encoding="utf-8") as f:
        f.write(f"Patient {i}: (row {idx}), Prediction: {prediction}\n\n")
        f.write(result)

    print(f"Saved: {file_path}")

Saved: Generative_AI/example_outputs/template_3_patient_1.txt
Saved: Generative_AI/example_outputs/template_3_patient_2.txt
Saved: Generative_AI/example_outputs/template_3_patient_3.txt


### Template 4: Step-by-Step Guide

**Goal:** Walk through each lab value one by one to explain why the model made its prediction.

**Target user:** Patients who want to understand their result in detail, and doctors who need a transparent review of the reasoning.

**Design choice:** Step-by-step breakdown of each biomarker against its normal range, followed by a reasoning conclusion that ties everything together.

In [5]:
# Template 4: Step-by-Step Guide
# Goal: walk through each lab value and explain the reasoning behind the prediction

template_4_system = """You are a medical reasoning assistant.
When given lab values and a prediction, think step by step:
1. Check each value against its normal range.
2. Note which ones are too high, too low, or normal.
3. Explain how the abnormal values connect to the prediction.
4. End with a short conclusion that ties everything together.
Keep your language clear and use the reference ranges as part of your reasoning."""

template_4_user = """A machine learning model made the following prediction for a liver patient.

Prediction: {prediction}
Lab Values:
{features}

Go through each lab value step by step and explain whether it supports
or goes against the prediction. End with a clear reasoning conclusion."""


In [6]:
def step_by_step_guide(prediction, features):
    """Generate a step-by-step reasoning analysis using Template 4."""

    user_message = template_4_user.format(
        prediction=prediction,
        features=features
    )

    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": template_4_system},
            {"role": "user", "content": user_message}
        ],
        temperature=0.3,
        max_tokens=600
    )

    return response.choices[0].message.content


In [7]:
# Test Template 4 0n 3 test cases

for i, idx in enumerate(test_indices, start=1):
    patient = data.iloc[idx]

    prediction = "Liver Disease Detected" if patient["Dataset"] == 1 else "No Liver Disease"

    features = "\n".join([
        f"- {col}: {val}"
        for col, val in patient.drop("Dataset").items()
    ])

    print(f"Patient {i}: (row {idx}), Prediction: {prediction}")

    result = step_by_step_guide(prediction, features)
    print(result)


Patient 1: (row 0), Prediction: Liver Disease Detected
To evaluate the prediction of "Liver Disease Detected," let's examine each lab value against its normal range and see how it supports or contradicts the prediction.

1. **Age: 65** - Age itself is not a lab value but a demographic factor. However, it's known that the risk of liver disease can increase with age. Thus, being 65 might slightly increase the likelihood of liver disease, but it's not a direct indicator.

2. **Gender: Female** - Like age, gender is a demographic factor and not a lab value. Some liver diseases have different prevalence rates among genders, but without specific context, it's hard to draw a direct connection to the prediction.

3. **Total_Bilirubin: 0.7** - The normal range for total bilirubin is approximately 0.1 to 1.2 mg/dL. With a value of 0.7, this falls within the normal range. Normal bilirubin levels do not typically indicate liver disease, so this value does not support the prediction.

4. **Direct_B

# 4. Selection & Justification

## Selected Template: **T2 - Personalized Explanation**

After reviewing the analysis, **Template 2** was selected as the best prompt for the final system because it offers the strongest balance between personalization, clarity, and clinical relevance for an advice system aimed at patients.

---

### Qualitative Strengths

Based on the qualitative evaluation:

- T2 is the **only template that scored High on Personalization**, which is essential for an advice system that should respond to each patient's specific values rather than producing a generic message.
- T2 scored **High on Clarity**, second only to T1, meaning the explanations stay readable for patients without medical training.
- T2 scored **High on Relevance**, so the responses connect directly to the prediction without going off-topic.
- T2 maintains **High Safety**, the same level as the other templates, meaning the personalization does not come at the cost of unsafe communication.
- The analysis described T2 as more *engaging and meaningful* than T1, while avoiding the heavier structure of T3 and T4.

T1 is too low on Detail and Personalization to count as real advice. T3 and T4 score higher on Detail but lose out on Clarity and Personalization, making them less suitable for the patient-facing advice system our project is building.

---

### Quantitative Metrics

Based on the quantitative analysis:

- **Response Length**: T2 averages **200 words** - long enough to be informative but shorter than T3 (220) and T4 (230). T1 at 80 words is too brief to provide real advice.
- **Keyword Usage**: T2 uses **8-10 domain keywords** on average, which is enough to be clinically grounded without being overloaded like T4 (15-18 keywords).
- **Readability**: T2 is rated **Easy**, more accessible than T3 (Medium) or T4 (Hard). This matches the target audience (patients) without medical training. Only T1 is easier (Very Easy), but T1 lacks the necessary detail.

T2 is the only template that hits a balanced position across all three quantitative dimensions: not too short, clinically relevant, and easy to read.

---

### Alignment with System Goals

The system's goal is to provide an **advice system** that helps users understand their liver screening result. The three core requirements for this are:

1. **Personalization** - each user should get an explanation tied to their own data → T2 is the strongest template on this dimension (rated High while others are Medium or Low).
2. **Clinical Relevance** - outputs must reference real medical content → T2 uses 8–10 domain keywords, appropriate without being overwhelming.
3. **Accessibility** - the explanation must be understandable by non-experts → T2 has High Clarity and Easy readability, fitting the target audience.

T2 is the only template that scores well across all three of these requirements at the same time. T1 fails on personalization, T3 and T4 fall short on clarity and accessibility for non-expert users.

---

## Limitations, Ethical Considerations & Potential Improvements

### Limitations

- T2 may be too long for users who only want a one-line answer.
- The personalization quality depends on the LLM correctly identifying which patient values are most relevant, this can be inconsistent for borderline cases.
- T2 does not include an explicit step-by-step reasoning conclusion like T4 does, so users who want full interpretability would need to look at T4.
- The Easy readability comes partly from avoiding specific reference ranges, which means users do not see the exact thresholds the model is reasoning against.

### Ethical Considerations

- Every output should include a disclaimer that the result is generated by an AI system and is not a real medical diagnosis.
- The Indian Liver Patient Dataset is skewed toward male patients aged 30–65, so predictions and explanations may be less accurate for elderly women, younger patients, or other underrepresented groups.
- There is a risk that users trust the AI's response too much and skip a real doctor visit, which could delay proper care.

### Potential Improvements

- Add age and gender awareness into the system prompt so the explanation adjusts naturally based on who the patient is.
- Add a verification step that checks any LLM-stated reference ranges against a trusted medical source before showing the output (especially relevant if T3 or T4 are used in the future).
- Try few-shot prompting where one good worked example is given to the model for each prediction class, which can improve consistency on edge cases.
- Build a hybrid setup where T2 is the main view but the user can click "Show full reasoning" to see the T4 step-by-step output for more detail.
- Display the supervised model's confidence score alongside the explanation so users know how certain the prediction actually is.